In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import os, re, json

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
RAW = ROOT / 'datasets' / 'raw'
PROC = ROOT / 'datasets' / 'processed'

for p in [
    PROC / 'module1_bus',
    PROC / 'module2_breakdown',
    PROC / 'module3_uturn',
    PROC / 'shared',
]:
    p.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('RAW exists:', RAW.exists())

Mounted at /content/drive
ROOT: /content/drive/MyDrive/TrafficAI_ControlCenter
RAW exists: True


In [ ]:
!pip -q install openpyxl pyarrow

In [ ]:
gtfs = next((RAW / 'delhi_gtfs').rglob('stops.txt')).parent
gps_root = RAW / 'delhi_bus_gps'
ai_root = RAW / 'aicity2021_t4'
label_file = next(ai_root.rglob('train-anomaly-results.txt'))
videos = list(ai_root.rglob('*.mp4'))

print('GTFS folder:', gtfs)
print('GPS excel count:', len(list(gps_root.rglob('*.xlsx'))))
print('AI label file:', label_file)
print('AI videos:', len(videos))

GTFS folder: /content/drive/MyDrive/TrafficAI_ControlCenter/datasets/raw/delhi_gtfs/extracted
GPS excel count: 184
AI label file: /content/drive/MyDrive/TrafficAI_ControlCenter/datasets/raw/aicity2021_t4/AIC21-Track4-Anomaly-Detection/train-anomaly-results.txt
AI videos: 250


In [ ]:
sample = next((RAW / 'delhi_bus_gps').rglob('*.xlsx'))
print('Sample file:', sample)

df = pd.read_excel(sample)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
display(df.head(10))

Sample file: /content/drive/MyDrive/TrafficAI_ControlCenter/datasets/raw/delhi_bus_gps/Month_3/425/Day1.xlsx
Shape: (1750, 5)
Columns: ['Unnamed: 0', 'Node', 'Number', 'Route', 'Time']


,Unnamed: 0,Node,Number,Route,Time
0,0,17,DL1PC 5377,up,05:14:59:000
1,1,18,DL1PC 5377,up,05:16:59:000
2,2,19,DL1PC 5377,up,05:17:49:000
3,3,20,DL1PC 5377,up,05:18:59:000
4,4,21,DL1PC 5377,up,05:19:49:000
5,5,22,DL1PC 5377,up,05:20:29:000
6,6,23,DL1PC 5377,up,05:21:49:000
7,7,25,DL1PC 5377,up,05:23:23:000
8,8,26,DL1PC 5377,up,05:24:43:000
9,9,28,DL1PC 5377,up,05:26:03:000


In [ ]:
gps_files = sorted((RAW / 'delhi_bus_gps').rglob('*.xlsx'))
print('Reading', len(gps_files), 'excel files...')

rows = []
for i, f in enumerate(gps_files):
    try:
        d = pd.read_excel(f)
    except Exception as e:
        print('Skip', f.name, e)
        continue

    d.columns = [str(c).strip() for c in d.columns]
    lower = {c.lower(): c for c in d.columns}

    # Flexible column mapping (Indian GPS dumps vary)
    def pick(*cands):
        for c in cands:
            if c in lower:
                return lower[c]
        return None

    c_lat = pick('lat', 'latitude', 'y', 'gps_lat')
    c_lon = pick('lon', 'lng', 'longitude', 'x', 'gps_lon', 'long')
    c_time = pick('time', 'timestamp', 'datetime', 'date_time', 'gps_time', 'dt')
    c_speed = pick('speed', 'velocity', 'spd', 'speed_kmph', 'speed_km/h')

    # route/day from path: Month_3/425/Day1.xlsx
    parts = f.parts
    route_id = None
    for p in parts:
        if p.isdigit() and len(p) <= 4:
            route_id = p
    day = f.stem  # Day1
    month = next((p for p in parts if p.lower().startswith('month')), None)

    if c_lat is None or c_lon is None:
        if i == 0:
            print('Could not find lat/lon in columns:', list(d.columns))
        continue

    tmp = pd.DataFrame({
        'route_id': route_id,
        'month': month,
        'day': day,
        'source_file': str(f.relative_to(RAW / 'delhi_bus_gps')),
        'lat': pd.to_numeric(d[c_lat], errors='coerce'),
        'lon': pd.to_numeric(d[c_lon], errors='coerce'),
    })
    if c_time:
        tmp['timestamp_raw'] = d[c_time].astype(str)
    else:
        tmp['timestamp_raw'] = None
    if c_speed:
        tmp['speed'] = pd.to_numeric(d[c_speed], errors='coerce')
    else:
        tmp['speed'] = np.nan

    rows.append(tmp)
    if (i + 1) % 20 == 0:
        print('  processed', i + 1, '/', len(gps_files))

bus_gps = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
bus_gps = bus_gps.dropna(subset=['lat', 'lon'])
print('Module1 GPS rows:', len(bus_gps))
print(bus_gps.head())

out1 = PROC / 'module1_bus' / 'bus_gps_points.parquet'
bus_gps.to_parquet(out1, index=False)
bus_gps.head(5000).to_csv(PROC / 'module1_bus' / 'bus_gps_points_sample.csv', index=False)
print('Saved:', out1)

Reading 184 excel files...
Could not find lat/lon in columns: ['Unnamed: 0', 'Node', 'Number', 'Route', 'Time']


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import pandas as pd

gps_root = Path('/content/drive/MyDrive/TrafficAI_ControlCenter/datasets/raw/delhi_bus_gps')

# list structure
for p in sorted(gps_root.rglob('*'))[:50]:
    if p.is_file():
        print(p.relative_to(gps_root))

# search any csv/xlsx that might contain lat/lon
hits = []
for f in gps_root.rglob('*'):
    if f.suffix.lower() not in ['.csv', '.xlsx', '.xls', '.txt', '.json']:
        continue
    try:
        if f.suffix.lower() == '.csv':
            d = pd.read_csv(f, nrows=5)
        elif f.suffix.lower() in ['.xlsx', '.xls']:
            d = pd.read_excel(f, nrows=5)
        else:
            continue
        cols = ' '.join(map(str, d.columns)).lower()
        if any(k in cols for k in ['lat', 'lon', 'lng', 'latitude', 'longitude', 'coord']):
            hits.append((str(f.relative_to(gps_root)), list(d.columns)))
    except Exception:
        pass

print('Lat/lon-like files found:', hits if hits else 'NONE')

Month_1/425/Day1.xlsx
Month_1/425/Day10.xlsx
Month_1/425/Day11.xlsx
Month_1/425/Day12.xlsx
Month_1/425/Day13.xlsx
Month_1/425/Day14.xlsx
Month_1/425/Day15.xlsx
Month_1/425/Day16.xlsx
Month_1/425/Day17.xlsx
Month_1/425/Day18.xlsx
Month_1/425/Day19.xlsx
Month_1/425/Day2.xlsx
Month_1/425/Day20.xlsx
Month_1/425/Day21.xlsx
Month_1/425/Day22.xlsx
Month_1/425/Day23.xlsx
Month_1/425/Day24.xlsx
Month_1/425/Day25.xlsx
Month_1/425/Day26.xlsx
Month_1/425/Day27.xlsx
Month_1/425/Day28.xlsx
Month_1/425/Day29.xlsx
Month_1/425/Day3.xlsx
Month_1/425/Day30.xlsx
Month_1/425/Day31.xlsx
Month_1/425/Day4.xlsx
Month_1/425/Day5.xlsx
Month_1/425/Day6.xlsx
Month_1/425/Day7.xlsx
Month_1/425/Day8.xlsx
Month_1/425/Day9.xlsx
Month_1/534/Day1.xlsx
Month_1/534/Day10.xlsx
Month_1/534/Day11.xlsx
Month_1/534/Day12.xlsx
Month_1/534/Day13.xlsx
Month_1/534/Day14.xlsx
Month_1/534/Day15.xlsx
Month_1/534/Day16.xlsx
Month_1/534/Day17.xlsx
Month_1/534/Day18.xlsx
Month_1/534/Day19.xlsx
Month_1/534/Day2.xlsx
Month_1/534/Day20.xlsx

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
RAW = ROOT / 'datasets' / 'raw'
PROC = ROOT / 'datasets' / 'processed' / 'module1_bus'
PROC.mkdir(parents=True, exist_ok=True)

# --- GTFS ---
gtfs_dir = next((RAW / 'delhi_gtfs').rglob('stops.txt')).parent
stops = pd.read_csv(gtfs_dir / 'stops.txt')
routes = pd.read_csv(gtfs_dir / 'routes.txt')
trips = pd.read_csv(gtfs_dir / 'trips.txt')
stop_times = pd.read_csv(gtfs_dir / 'stop_times.txt')

print('GTFS loaded')

# Find route 425 / 534 in GTFS (short_name or long_name)
routes['route_short_name'] = routes.get('route_short_name', pd.Series(dtype=str)).astype(str)
routes['route_long_name'] = routes.get('route_long_name', pd.Series(dtype=str)).astype(str)

def find_route_ids(code):
    m = routes[
        routes['route_short_name'].str.contains(str(code), na=False)
        | routes['route_long_name'].str.contains(str(code), na=False)
    ]
    return m[['route_id', 'route_short_name', 'route_long_name']]

print('Route 425 matches:\n', find_route_ids(425))
print('Route 534 matches:\n', find_route_ids(534))

# Build stop sequence table for a route_id: node_index = stop_sequence order
# GTFS stop_sequence is 1..N along a trip; your Excel "Node" looks like that index.

def build_node_map(route_code):
    cand = find_route_ids(route_code)
    if cand.empty:
        return None, None
    rid = cand.iloc[0]['route_id']
    trip_ids = trips.loc[trips['route_id'] == rid, 'trip_id']
    st = stop_times[stop_times['trip_id'].isin(trip_ids)].copy()
    # use the most common trip shape/sequence
    seq_counts = st.groupby('trip_id').size().sort_values(ascending=False)
    best_trip = seq_counts.index[0]
    seq = st[st['trip_id'] == best_trip][['stop_sequence', 'stop_id']].drop_duplicates()
    seq = seq.sort_values('stop_sequence')
    seq = seq.merge(stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id', how='left')
    # map Node ~= stop_sequence (and also try Node as 0-based / rank)
    seq['node_as_sequence'] = seq['stop_sequence']
    seq['node_as_rank'] = np.arange(1, len(seq) + 1)
    return rid, seq

# --- Read all GPS excels as node logs ---
gps_files = sorted((RAW / 'delhi_bus_gps').rglob('*.xlsx'))
rows = []
for i, f in enumerate(gps_files):
    d = pd.read_excel(f)
    d.columns = [str(c).strip() for c in d.columns]
    # path: Month_3/425/Day1.xlsx
    parts = f.parts
    route_code = None
    for p in parts:
        if re.fullmatch(r'\d{2,4}', p):
            route_code = int(p)
            break
    month = next((p for p in parts if str(p).lower().startswith('month')), None)
    tmp = pd.DataFrame({
        'route_code': route_code,
        'month': month,
        'day': f.stem,
        'node': pd.to_numeric(d['Node'], errors='coerce'),
        'bus_number': d['Number'].astype(str),
        'direction': d['Route'].astype(str),
        'time_raw': d['Time'].astype(str),
        'source_file': str(f.relative_to(RAW / 'delhi_bus_gps')),
    })
    rows.append(tmp)
    if (i + 1) % 30 == 0:
        print('read', i + 1, '/', len(gps_files))

logs = pd.concat(rows, ignore_index=True).dropna(subset=['node'])
logs['node'] = logs['node'].astype(int)
print('Log rows:', len(logs))
print(logs.head())

# Build maps for routes present
node_maps = {}
for code in sorted(logs['route_code'].dropna().unique()):
    rid, seq = build_node_map(int(code))
    print('Route', code, 'GTFS route_id', rid, 'stops in sequence', 0 if seq is None else len(seq))
    if seq is not None:
        node_maps[int(code)] = seq
        seq.to_csv(PROC / f'gtfs_node_map_route_{code}.csv', index=False)

# Join: try node == stop_sequence, fallback node == rank
joined_parts = []
for code, part in logs.groupby('route_code'):
    seq = node_maps.get(int(code))
    if seq is None:
        print('No GTFS map for route', code, '- keeping without lat/lon')
        part = part.copy()
        part['stop_lat'] = np.nan
        part['stop_lon'] = np.nan
        part['stop_id'] = None
        part['stop_name'] = None
        joined_parts.append(part)
        continue
    p1 = part.merge(seq, left_on='node', right_on='node_as_sequence', how='left')
    # fill misses via rank
    miss = p1['stop_lat'].isna()
    if miss.any():
        p2 = part.loc[miss.values].merge(seq, left_on='node', right_on='node_as_rank', how='left', suffixes=('', '_r'))
        for col in ['stop_id', 'stop_name', 'stop_lat', 'stop_lon']:
            p1.loc[miss, col] = p2[col].values
    joined_parts.append(p1)

bus = pd.concat(joined_parts, ignore_index=True)
print('Joined rows:', len(bus))
print('With coordinates:', bus['stop_lat'].notna().sum(), '/', len(bus))
print(bus[['route_code','bus_number','node','time_raw','stop_name','stop_lat','stop_lon']].head(10))

bus.to_parquet(PROC / 'bus_gps_points.parquet', index=False)
bus.head(8000).to_csv(PROC / 'bus_gps_points_sample.csv', index=False)
print('Saved processed Module1 table')

GTFS loaded
Route 425 matches:
       route_id route_short_name route_long_name
92          92              nan     425LNKSTLUP
375        375              nan   425LNKSTLDOWN
396        396              nan      425STLDOWN
700        700              nan        425STLUP
1279      1279              nan       425STLUP2
1348      1348              nan     425STLDOWN2
1416      1416              nan         425CLUP
1946      1946              nan       425CLDOWN
2132      5168              nan           425UP
2599      5635              nan         425DOWN
Route 534 matches:
       route_id route_short_name route_long_name
281        281              nan      534CLVIAUP
435        435              nan          534BUP
1231      1231              nan        534EXTUP
1479      1479              nan       534UPVIA2
1513      1513              nan      534EXTDOWN
1656      1656              nan           534UP
1719      1719              nan         534DOWN
1889      1889              nan     

In [8]:
gtfs_dir = next((RAW / 'delhi_gtfs').rglob('stops.txt')).parent

stops = pd.read_csv(gtfs_dir / 'stops.txt')
routes = pd.read_csv(gtfs_dir / 'routes.txt')
trips = pd.read_csv(gtfs_dir / 'trips.txt')
stop_times = pd.read_csv(gtfs_dir / 'stop_times.txt')

print('stops', stops.shape, 'routes', routes.shape, 'trips', trips.shape, 'stop_times', stop_times.shape)

stops.to_parquet(PROC / 'module1_bus' / 'gtfs_stops.parquet', index=False)
routes.to_parquet(PROC / 'module1_bus' / 'gtfs_routes.parquet', index=False)
trips.to_parquet(PROC / 'module1_bus' / 'gtfs_trips.parquet', index=False)
# stop_times can be large; parquet is fine on Drive
stop_times.to_parquet(PROC / 'module1_bus' / 'gtfs_stop_times.parquet', index=False)
print('GTFS saved under processed/module1_bus/')

stops (6342, 6) routes (2964, 5) trips (78515, 4) stop_times (3276092, 5)


OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive/TrafficAI_ControlCenter/datasets/processed/module1_bus/module1_bus'

In [9]:
from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
RAW = ROOT / 'datasets' / 'raw'
PROC = ROOT / 'datasets' / 'processed' / 'module1_bus'
PROC.mkdir(parents=True, exist_ok=True)

gtfs_dir = next((RAW / 'delhi_gtfs').rglob('stops.txt')).parent
stops = pd.read_csv(gtfs_dir / 'stops.txt')
routes = pd.read_csv(gtfs_dir / 'routes.txt')
trips = pd.read_csv(gtfs_dir / 'trips.txt')
stop_times = pd.read_csv(gtfs_dir / 'stop_times.txt')

print('stops', stops.shape, 'routes', routes.shape, 'trips', trips.shape, 'stop_times', stop_times.shape)

stops.to_parquet(PROC / 'gtfs_stops.parquet', index=False)
routes.to_parquet(PROC / 'gtfs_routes.parquet', index=False)
trips.to_parquet(PROC / 'gtfs_trips.parquet', index=False)
stop_times.to_parquet(PROC / 'gtfs_stop_times.parquet', index=False)
print('GTFS saved to', PROC)

stops (6342, 6) routes (2964, 5) trips (78515, 4) stop_times (3276092, 5)
GTFS saved to /content/drive/MyDrive/TrafficAI_ControlCenter/datasets/processed/module1_bus


In [10]:
label_file = next((RAW / 'aicity2021_t4').rglob('train-anomaly-results.txt'))
print('Reading', label_file)

# Official format: video_id start_sec end_sec  (space-separated)
anom = pd.read_csv(
    label_file,
    sep=r'\s+',
    header=None,
    names=['video_id', 'start_sec', 'end_sec'],
    engine='python'
)
print(anom.head())
print('Anomaly events:', len(anom))

# Map video_id -> mp4 path
videos = {
    int(p.stem): p
    for p in (RAW / 'aicity2021_t4').rglob('*.mp4')
    if p.stem.isdigit()
}

records = []
for _, r in anom.iterrows():
    vid = int(r['video_id'])
    vp = videos.get(vid)
    records.append({
        'video_id': vid,
        'start_sec': float(r['start_sec']),
        'end_sec': float(r['end_sec']),
        'duration_sec': float(r['end_sec']) - float(r['start_sec']),
        'video_path': str(vp) if vp else None,
        'label': 1,  # anomaly / incident-like
    })

m2 = pd.DataFrame(records)
print('Matched videos:', m2['video_path'].notna().sum(), '/', len(m2))
m2.to_parquet(PROC / 'module2_breakdown' / 'anomaly_events.parquet', index=False)
m2.to_csv(PROC / 'module2_breakdown' / 'anomaly_events.csv', index=False)
print('Saved module2 anomaly index')

Reading /content/drive/MyDrive/TrafficAI_ControlCenter/datasets/raw/aicity2021_t4/AIC21-Track4-Anomaly-Detection/train-anomaly-results.txt
   video_id  start_sec  end_sec
0         2        587      894
1         9          0      287
2        11          0      888
3        14        475      600
4        33          0       90
Anomaly events: 18
Matched videos: 18 / 18


OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive/TrafficAI_ControlCenter/datasets/processed/module1_bus/module2_breakdown'

In [11]:
from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
RAW = ROOT / 'datasets' / 'raw'
PROC = ROOT / 'datasets' / 'processed'   # <-- root processed, NOT module1_bus
out_dir = PROC / 'module2_breakdown'
out_dir.mkdir(parents=True, exist_ok=True)

label_file = next((RAW / 'aicity2021_t4').rglob('train-anomaly-results.txt'))
anom = pd.read_csv(
    label_file, sep=r'\s+', header=None,
    names=['video_id', 'start_sec', 'end_sec'], engine='python'
)

videos = {
    int(p.stem): p
    for p in (RAW / 'aicity2021_t4').rglob('*.mp4')
    if p.stem.isdigit()
}

records = []
for _, r in anom.iterrows():
    vid = int(r['video_id'])
    vp = videos.get(vid)
    records.append({
        'video_id': vid,
        'start_sec': float(r['start_sec']),
        'end_sec': float(r['end_sec']),
        'duration_sec': float(r['end_sec']) - float(r['start_sec']),
        'video_path': str(vp) if vp else None,
        'label': 1,
    })

m2 = pd.DataFrame(records)
print('Matched videos:', m2['video_path'].notna().sum(), '/', len(m2))

m2.to_parquet(out_dir / 'anomaly_events.parquet', index=False)
m2.to_csv(out_dir / 'anomaly_events.csv', index=False)
print('Saved to', out_dir)

Matched videos: 18 / 18
Saved to /content/drive/MyDrive/TrafficAI_ControlCenter/datasets/processed/module2_breakdown


In [12]:
from pathlib import Path

ngsim_path = Path(
    '/content/drive/MyDrive/TrafficAI_ControlCenter/datasets/raw/ngsim/ngsim_all.csv'
)

if ngsim_path.exists():
    print("NGSIM file found!")
    print("File size:", round(ngsim_path.stat().st_size / (1024**3), 3), "GB")
else:
    print("NGSIM file NOT found!")

NGSIM file found!
File size: 1.973 GB


In [13]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
RAW = ROOT / 'datasets' / 'raw'
PROC = ROOT / 'datasets' / 'processed'
out_dir = PROC / 'module3_uturn'
out_dir.mkdir(parents=True, exist_ok=True)

ngsim_path = RAW / 'ngsim' / 'ngsim_all.csv'
print('File size GB:', round(ngsim_path.stat().st_size / 1e9, 3))

# CPU-safe: first 250k rows only
NROWS = 250000
ngsim = pd.read_csv(ngsim_path, nrows=NROWS, low_memory=False)

print('Loaded sample shape:', ngsim.shape)
print('Columns:', list(ngsim.columns))


File size GB: 2.118
Loaded sample shape: (250000, 25)
Columns: ['Vehicle_ID', 'Frame_ID', 'Total_Frames', 'Global_Time', 'Local_X', 'Local_Y', 'Global_X', 'Global_Y', 'v_length', 'v_Width', 'v_Class', 'v_Vel', 'v_Acc', 'Lane_ID', 'O_Zone', 'D_Zone', 'Int_ID', 'Section_ID', 'Direction', 'Movement', 'Preceding', 'Following', 'Space_Headway', 'Time_Headway', 'Location']


In [14]:

# Identify NGSIM columns
cols = {c.lower(): c for c in ngsim.columns}

def col(*names):
    for n in names:
        if n.lower() in cols:
            return cols[n.lower()]
    return None

c_veh = col('Vehicle_ID')
c_frame = col('Frame_ID')
c_x = col('Local_X')
c_y = col('Local_Y')
c_lane = col('Lane_ID')
c_vel = col('v_Vel')

need = [c_veh, c_frame, c_x, c_y]
print('Mapped:', dict(
    veh=c_veh,
    frame=c_frame,
    x=c_x,
    y=c_y,
    lane=c_lane,
    vel=c_vel
))

assert all(need), 'Unexpected NGSIM columns'

# Prepare vehicle tracks
df = ngsim[
    [c_veh, c_frame, c_x, c_y, c_lane, c_vel]
].copy()

df = df.rename(columns={
    c_veh: 'vehicle_id',
    c_frame: 'frame_id',
    c_x: 'x',
    c_y: 'y',
    c_lane: 'lane_id',
    c_vel: 'speed'
})

# Sort vehicle positions by frame
df = df.sort_values(['vehicle_id', 'frame_id'])

# Calculate movement
df['dx'] = df.groupby('vehicle_id')['x'].diff()
df['dy'] = df.groupby('vehicle_id')['y'].diff()

# Calculate heading in radians
df['heading'] = np.arctan2(df['dy'], df['dx'])

# Unwrap heading changes
df['heading_unwrap'] = df.groupby('vehicle_id')['heading'].transform(
    lambda s: np.unwrap(s.fillna(0).to_numpy())
)

df['heading_delta'] = df.groupby('vehicle_id')['heading_unwrap'].diff()

# Identify preliminary U-turn candidates
veh_stats = df.groupby('vehicle_id').agg(
    max_abs_turn=(
        'heading_delta',
        lambda s: np.nanmax(np.abs(s)) if s.notna().any() else np.nan
    ),
    n_frames=('frame_id', 'count'),
    mean_speed=('speed', 'mean')
).reset_index()

veh_stats['uturn_candidate'] = (
    veh_stats['max_abs_turn'] > 2.5
).astype(int)

print('\nU-turn candidate results:')
print(veh_stats['uturn_candidate'].value_counts(dropna=False))
print('\nTotal vehicles:', len(veh_stats))
print('U-turn candidates:', int(veh_stats['uturn_candidate'].sum()))


Mapped: {'veh': 'Vehicle_ID', 'frame': 'Frame_ID', 'x': 'Local_X', 'y': 'Local_Y', 'lane': 'Lane_ID', 'vel': 'v_Vel'}

U-turn candidate results:
uturn_candidate
1    613
0    489
Name: count, dtype: int64

Total vehicles: 1102
U-turn candidates: 613


In [15]:

# Save processed NGSIM tracks and candidate labels

out_tracks = PROC / 'module3_uturn' / 'ngsim_tracks_sample.parquet'
out_labels = PROC / 'module3_uturn' / 'uturn_candidate_labels.csv'

df.head(250000).to_parquet(out_tracks, index=False)
veh_stats.to_csv(out_labels, index=False)

print('Saved track data:', out_tracks)
print('Saved candidate labels:', out_labels)
print('U-turn candidates:', int(veh_stats['uturn_candidate'].sum()), '/', len(veh_stats))
print('\nSTEP 3.6 COMPLETED!')


Saved track data: /content/drive/MyDrive/TrafficAI_ControlCenter/datasets/processed/module3_uturn/ngsim_tracks_sample.parquet
Saved candidate labels: /content/drive/MyDrive/TrafficAI_ControlCenter/datasets/processed/module3_uturn/uturn_candidate_labels.csv
U-turn candidates: 613 / 1102

STEP 3.6 COMPLETED!


In [16]:
import json
from pathlib import Path
import pandas as pd

PROC = Path('/content/drive/MyDrive/TrafficAI_ControlCenter/datasets/processed')

def rows(path):
    p = Path(path)
    if not p.exists():
        return 0
    if p.suffix == '.parquet':
        return len(pd.read_parquet(p))
    return len(pd.read_csv(p))

manifest = {
    'module1_bus_gps_rows': rows(PROC/'module1_bus'/'bus_gps_points.parquet'),
    'module1_gtfs_stops': rows(PROC/'module1_bus'/'gtfs_stops.parquet'),
    'module2_anomaly_events': rows(PROC/'module2_breakdown'/'anomaly_events.csv'),
    'module3_uturn_candidates': rows(PROC/'module3_uturn'/'uturn_candidate_labels.csv'),
}
(PROC/'shared').mkdir(exist_ok=True)
(PROC/'shared'/'step3_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

ok = all(manifest[k] > 0 for k in manifest)
print('STEP 3 OK' if ok else 'STEP 3 INCOMPLETE')

{
  "module1_bus_gps_rows": 924375,
  "module1_gtfs_stops": 6342,
  "module2_anomaly_events": 18,
  "module3_uturn_candidates": 1102
}
STEP 3 OK
